# 00 · FMM-JEPA Core Theory and Verification

**Milestone:** establish a tested, reusable foundation for the six-notebook image-representation project. This notebook is a mathematical lab book and executable verification report, not a claim that FMM-JEPA learns useful representations.

**Inputs:** local tagged notebook modules, `configs/core_cpu_smoke.yaml` (default), synthetic images, fixed numerical fixtures. No real dataset or credentials. **Outputs:** certificates, raw timings, synthetic metrics, figures, parameter accounting, handoff and resumable checkpoints. **Expected runtime:** roughly 2–8 minutes for this bounded CPU run plus optional small CUDA measurements; hardware affects timing. No long training job is launched.

**Paths:** artifacts are written to `artifacts/notebook_00/<run_id>/`; authoritative checkpoints are in `checkpoints/<run_id>/`. The artifact directory includes a clearly labeled link to its final synthetic checkpoint. Checkpoint purpose is `synthetic_smoke_test`.

**Exact resume:** set `FMM_JEPA_ROOT` to this project directory and `FMM_JEPA_RESUME_RUN_ID` to an existing run ID, keep its original profile/configuration, then restart the kernel and Run All. The cell `run-synthetic-smoke` restores the manifest's latest valid checkpoint and verifies step/loss/config/source hashes and software/device/determinism settings. It never silently restarts on invalid state. Omit the resume variable for a new unique run. Set `FMM_JEPA_PROFILE=core_gpu_smoke` only to select the documented GPU synthetic profile. Set `FMM_JEPA_OPTIONAL_CUDA=1` to add the bounded GPU smoke/benchmark panel; CPU-only execution remains supported.

**Packaging decision:** the user retained the notebook-only working agreement. Shared modules live in `fmm_jepa/*.ipynb` and expose only `ahr-export` definition cells through an import hook. Ordinary `from fmm_jepa...` imports work after explicit bootstrap; importing does not execute an experiment. See [import contract](../docs/notebook_imports.md).

The final cells print the **actual** gate status and readiness from this run. Read their saved output before interpreting any claim.

## 1. Scope, roadmap and controlled hypothesis

Can a geometrically structured near/far token mixer retain useful visual representations while improving measured quality, compute and memory trade-offs? Correctness, learning, efficiency and scaling are four separate questions. This notebook certifies bounded software/numerical properties and measures a small implementation; the learning hypothesis remains open.

| Notebook | Responsibility | Gate/output |
|---|---|---|
| **00 · Core Theory and Verification** | Current assignment | Numerical/gradient/recovery certificates and handoff |
| 01 · TinyImageNet Debug | First real learning comparison | Frozen-encoder evaluation and learning gate |
| 02 · Food101 Main | Controlled prototype study | Quality, resolution and registered ablations |
| 03 · COCO Spatial | Defined multi-label/spatial protocol | Transfer plus bounded spatial diagnostics |
| 04 · ImageNet1K | Larger benchmark only after pilot justification | Registered, resumable benchmark |
| 05 · Benchmarks | Audit saved evidence | Quality–cost comparisons and limitations |

No Notebook 01–05 is created or run here. Later comparisons use fresh initialization unless explicitly labeled transfer. Synthetic smoke weights are not pretraining. Image upsampling adds tokens, not image information. Video, robotics, AHR planning and an FMM-native predictor are outside this milestone.

## 2. I-JEPA: predict latent targets, not pixels

Images have shape $[B,3,H,W]$. A non-overlapping patch embedding maps each patch to width $D$. Let $I$ be all original row-major patch indices, $C\subset I$ the visible context and $T_k\subset I$ target block $k$. The context sees only visible raw patch content; the teacher intentionally sees the complete image:

$$z_C=E_\theta(x_C;C),\quad z_{T_k}=[E_{\bar\theta}(x;I)]_{T_k},\quad \hat z_{T_k}=P_\phi(z_C,C,T_k).$$

The predictor uses visible latents and target positional/mask tokens, never true target latents. Each output is $[B,|T_k|,D]$ in the requested target order. For valid-mask $v_{bt}^{(k)}$, this implementation uses

$$\mathcal L=\frac{1}{KB}\sum_{k,b}\frac{\sum_{t,i}v_{bt}^{(k)}\,\operatorname{SmoothL1}_{\beta=1}(\hat z_{bti}^{(k)},\operatorname{stopgrad}[\operatorname{LN}(z_{bti}^{(k)})])}{D\sum_t v_{bt}^{(k)}}.$$

This weights samples and target blocks equally, and valid tokens equally within each sample/block. The teacher is an exact initialization copy, excluded from optimization and forced into evaluation mode. After a successful finite-gradient optimizer step, $\bar\theta\leftarrow m\bar\theta+(1-m)\theta$. Floating buffers use EMA; non-floating buffers are copied. There is no dropout or BatchNorm here.

The [original paper](https://arxiv.org/abs/2301.08243v3) describes squared L2; the [pinned official implementation](https://github.com/facebookresearch/ijepa/blob/52c1ae95d05f743e000e8f10a1f3a79b10cff048/src/train.py) uses SmoothL1 and feature LayerNorm. We follow that implementation choice. This is neither pixel reconstruction nor an InfoNCE objective.

## 3. Classical FMM and tensor-product interpolation

Order $p$ means **nodes per coordinate axis**, so a 2-D cell carries $r=p^2$ coefficients. Chebyshev nodes are $\xi_a=\cos((2a+1)\pi/(2p))$ up to ordering. Cardinal polynomials satisfy $\ell_a(\xi_b)=\delta_{ab}$. Their product $\ell_{ab}(x,y)=\ell_a(x)\ell_b(y)$ reproduces polynomials of degree less than $p$ separately in each coordinate.

For cell $c$ and scalar charges $q_j$, its moments are $M_{c,a}=\sum_{j\in c}\ell_a(\hat x_j)q_j$. A child-to-parent matrix $T_{q,a,b}=\ell_a(\hat\xi_{q,b})$ maps child moments **up** by $M_P=\sum_qT_qM_q$. The transpose maps parent local coefficients **down** by $L_q=T_q^\top L_P$. The common backend tests both identities.

At each level, a cell interacts with children of its parent's neighboring cells that are not its own neighbors. This gives at most 27 well-separated source cells per receiver in 2-D. Touching leaf cells exchange direct messages. Every source/receiver pair has exactly one owner.

The full far-field route is **P2M → M2M upward → M2L at every eligible level → L2L downward → L2P**. Coefficients are $[B,C_\ell,r,D]$ for learned features and $[B,C_\ell,r,1]$ for charges. The classical diagnostic approximates $\sum_{j\ne i}q_j/(\|x_i-x_j\|^2+\epsilon^2)^{1/2}$; near terms are direct and self is excluded. Only tiny diagnostics allocate dense pair tables/direct fields.

Reference scope: [kernel-independent FMM](https://doi.org/10.1016/j.jcp.2003.11.021), [Chebyshev black-box FMM](https://doi.org/10.1016/j.jcp.2009.08.031), and the versioned [source review](../docs/references.md). Classical interpolation accuracy is not a learned-model error bound.

## 4. The learned mixer and semantic-compression risk

Near edges include self and connect tokens in neighboring leaf cells. Their weights depend on receiver/source content and displacement, with bias-free value projections and valid-neighbor-count normalization. The far branch projects feature-valued moments, applies the same geometric transfers, learns $r\times r$ M2L matrices indexed by relative cell offset, evaluates local expansions, divides by visible-token count and optionally applies $\sigma(W_gh_i)$ at the receiver. These learned translations share across levels and channels; they do not reproduce an arbitrary attention kernel.

The residual block is $u=h+\operatorname{Mixer}(\operatorname{LN}(h))$, followed by $h'=u+\operatorname{MLP}(\operatorname{LN}(u))$. FMM and ViT use the same patch embedding, positional features and block scaffold; the baseline and predictor use PyTorch SDPA. Width equality does not imply parameter matching.

Geometry is independent of features. Rectangular grids retain coordinates $((c+.5)/W_p,(r+.5)/H_p)$. The quadtree pads the leaf lattice to a power-of-two square, zeroes absent content and traverses allocated empty cells. Spatial distance does not imply semantic similarity or compressibility; low-order moments can discard rare distant relationships. Receiver gates and residuals are hypotheses, not proofs. Semantic routing, exception edges and extra global attention remain future ablations.

With $N_g$ grid positions, $N_v$ visible tokens, width $D$, $r=p^2$, allocated cells $C$, leaf side $b$ (patches per leaf side), near edges $E_n\le9b^2N_g$ and far cell edges $E_f\le27C$, implemented costs include full-grid projections $O(BN_gD^2)$, near work $O(BE_nD)$, P2M/L2P $O(BN_grD)$, and transfers/M2L $O(B(C+E_f)r^2D)$. M2L edge temporaries also scale with $E_f r D$ and selected kernels with $E_fr^2$. The dependence on order is substantial: $r^2=p^4$. The cache stores geometry only and keys shape/order/leaf-size/device/dtype. No moments survive a forward call.

For bounded aspect ratio and fixed settings $C=O(N_g)$; for thin rectangular families the square padding can dominate. Cost is not solely proportional to $N_v$. Attention's pairwise component is $O(BN^2D)$, plus projections/MLPs. The attention predictor and teacher remain part of whole-JEPA cost, so no whole-system linear-time claim follows.

## 5. Environment, explicit profile and artifact run

This setup uses an absolute, overridable root and the one tagged loader. Profiles are JSON-compatible YAML, avoiding an extra parser dependency. Two CPU threads limit small-matrix scheduling noise. Sensitive numerical tests use float64; neural correctness uses float32. Installed PyTorch/CUDA are preserved. All evidence records the actual device/backend. The boot disk is persistent storage, but deletion policy remains unverified; export final/best checkpoints and verify hashes before recycling this instance.

In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
from pathlib import Path
import json, datetime, importlib.metadata, math, copy
PROJECT_ROOT = Path(os.environ.get("FMM_JEPA_ROOT", "/home/benjamin/workspace/ahr_go/fmm_jeppa")).expanduser().resolve()
_namespace = {}
for _cell in json.loads((PROJECT_ROOT/"fmm_jepa/bootstrap.ipynb").read_text())["cells"]:
    if _cell["cell_type"] == "code" and "ahr-export" in _cell.get("metadata", {}).get("tags", []):
        exec(compile("".join(_cell["source"]), "bootstrap.ipynb#ahr-export", "exec"), _namespace)
_namespace["install_notebook_package"](PROJECT_ROOT)
import torch
from fmm_jepa.training import *
from fmm_jepa.reporting import *
from fmm_jepa.evaluation import benchmark_sweep, cuda_smoke_test
from fmm_jepa.geometry import build_geometry, pair_owner
from fmm_jepa.models import JEPA
from fmm_jepa.data import sample_masks
from fmm_tests.test_geometry import run_geometry_tests
from fmm_tests.test_models import run_model_tests
from IPython.display import display, Markdown
PROFILE = os.environ.get("FMM_JEPA_PROFILE", "core_cpu_smoke")
if PROFILE not in ("core_cpu_smoke", "core_gpu_smoke"): raise ValueError("Unknown execution profile")
config = json.loads((PROJECT_ROOT/"configs"/(PROFILE+".yaml")).read_text())
torch.set_num_threads(config["cpu_threads"])
torch.use_deterministic_algorithms(True)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
seed_everything(config["seed"])
RUN_ID, ARTIFACT, CHECKPOINT_DIR, SOURCE_HASHES = prepare_run(PROJECT_ROOT, config, os.environ.get("FMM_JEPA_RESUME_RUN_ID"))
ENVIRONMENT = environment_report(PROJECT_ROOT, config)
atomic_json(ARTIFACT/"environment.json", ENVIRONMENT)
(ARTIFACT/"environment_packages.txt").write_text("\n".join(sorted(f"{d.metadata['Name']}=={d.version}" for d in importlib.metadata.distributions()))+"\n")
style_figures()
print("Run:", RUN_ID, "\nArtifacts:", ARTIFACT, "\nCheckpoints:", CHECKPOINT_DIR)
print(json.dumps(config, indent=2))
print(json.dumps(ENVIRONMENT, indent=2))


In [ ]:
save_figure(architecture_figure(), ARTIFACT, "architecture")

## 6. Patch geometry and reproducible multi-block masking

The dataset contract is `(image, sample_id, split, source_dimensions, optional labels/annotations)`; labels never enter the JEPA loss. This milestone generates synthetic images only. Each future adapter must pin its Hugging Face repository/revision/schema/access terms and explicit split mapping before use.

`sample_masks` samples rectangles with bounded retries. Target overlap is intentional; the union is removed from context. Its documented fallback reserves a non-target anchor so that the context stays nonempty. Indices are unique within each block. Variable lengths are padded with a boolean valid mask; original row-major indices are preserved. The context encoder gathers visible raw patches before embedding, so hidden content cannot enter a mixed feature. Invalid values, including NaNs, are zeroed before projection or message construction.

The displayed masks are actual samples, not a schematic. Model tests below perturb only hidden pixels and separately inject invalid padded values.

In [ ]:
grid = (config["image_height"]//config["model"]["patch_size"], config["image_width"]//config["model"]["patch_size"])
mask_generator = torch.Generator().manual_seed(config["seed"]+71)
example_masks = sample_masks(grid, config["batch_size"], mask_generator, **config["mask"])
save_figure(mask_figure(grid, example_masks), ARTIFACT, "sampled_masks")
print("Visible original indices, sample 0:", example_masks["context_indices"][0][example_masks["context_valid"][0]].tolist())

## 7. Hierarchy, exact ownership and numerical transfer certificates

Leaf size is patches per cell side. Parents group four children. Adjacent leaves own near pairs; the first well-separated ancestor pair owns each remaining interaction. Empty and out-of-image cells carry zero moments. Exhaustive tiny-grid ownership counts must equal one for every permitted ordered pair. The production path uses bounded sparse lists, never that dense certificate table.

These tests check rectangular/boundary cases, interpolation reproduction, transfer directions, scalar fields, nonzero near/distant paths, NaN padding isolation, finite-difference gradients and determinism. Structural counts plus forward/backward tensor-shape tracing inspect the actual production operator, beyond searching for a literal `N*N` expression.

In [ ]:
save_figure(hierarchy_figure(), ARTIFACT, "hierarchy")

In [ ]:
geometry_results = run_geometry_tests()
atomic_json(ARTIFACT/"geometry_certificates.json", geometry_results)
print(json.dumps({name:entry["status"] for name,entry in geometry_results.items()}, indent=2))
print("Current hierarchy:", json.dumps(build_geometry(grid, config["model"]["order"], config["model"]["leaf_size"]).stats, indent=2))

## 8. Direct-versus-hierarchical classical field

The fixed charge fixtures compare orders 2, 4 and 6. The absolute L2 error and stabilized relative error `||u_h-u_exact|| / max(||u_exact||, delta)` are retained in the raw certificate. The order-specific tolerances are declared in the test. Increasing order is investigated, not assumed to improve every signed-charge or finite-precision realization. These results concern the regularized scalar kernel only.

In [ ]:
classical_cases = geometry_results["classical_field_approximation"].get("details", {}).get("cases", [])
if classical_cases:
    save_figure(classical_figure(classical_cases), ARTIFACT, "classical_error")
    print(json.dumps(classical_cases, indent=2))
else:
    print("Classical diagnostic failed; inspect geometry_certificates.json")

## 9. Encoder, predictor, gradient isolation and EMA certificates

`VisionEncoder(kind='fmm'|'vit', ...)` returns features in supplied original-index order. A missing index argument requests the complete grid. `JEPA(config['model'])` exposes `context`, `target` and `predictor`; its forward result contains scalar loss, predictions, teacher targets, context features and detached diagnostics.

The tests certify shape/position correspondence, equal-block masked loss reduction, finite forward/backward/update, expected nonzero student/predictor gradients, absent teacher gradients, exact initial teacher copy, numerical EMA, deterministic execution and observed optimized SDPA dispatch. A matching tensor shape alone would not establish alignment or isolation.

In [ ]:
model_results = run_model_tests()
atomic_json(ARTIFACT/"model_certificates.json", model_results)
for name, entry in model_results.items(): print(entry["status"], name)
print("Detailed tolerances and measurements: model_certificates.json")

## 10. Checkpoint interruption and exact continuation

The test compares four uninterrupted steps with two steps, a complete model/optimizer/scheduler reconstruction and checkpoint restoration, and two more steps. It compares model state, optimizer state, scheduler state, outputs, the loss sequence and Python/NumPy/Torch/data-mask RNGs. A copied checkpoint is deliberately corrupted to ensure loading refuses it. The original valid checkpoints remain intact.

Writes use an advisory single-writer lock, an fsynced temporary checkpoint, atomic rename, and an atomically published manifest. Latest two and best are retained. Every synthetic step is a stage boundary and is checkpointed, exceeding the workspace's 15-minute/250-step minimum. No mixed-precision scaler or distributed replay is active; their disabled/single-process status is saved explicitly. Exact recovery is certified for this CPU software/hardware configuration, not every possible backend.

In [ ]:
try:
    checkpoint_results = checkpoint_resume_test(PROJECT_ROOT/"checkpoints", config, SOURCE_HASHES)
except Exception as error:
    import traceback
    checkpoint_results = {"checkpoint_exact_resume":{"status":"FAIL","details":{"error":str(error),"traceback":traceback.format_exc()}}}
atomic_json(ARTIFACT/"checkpoint_certificates.json", checkpoint_results)
print(json.dumps(checkpoint_results, indent=2))

## 11. A selected query's information path

Left: which near or far level owns every source-to-query interaction? Right: the measured gradient norm of one untrained query output with respect to each source feature. Distant nonzero sensitivity confirms a computational route; it does not show meaningful semantic retention. The cyan × is the receiver. These figures use the same real mixer and ownership logic tested above.

In [ ]:
save_figure(query_figure(), ARTIFACT, "query_information_flow")

## 12. Optimized attention baseline and initial scaling instrumentation

The default CPU sweep uses 16, 64 and 192 full-grid positions. Both mixers receive exactly the same full-token workload; full-JEPA rows use identical seeded masks and the full-grid teacher. Each scope is labeled separately: mixer, residual encoder block, or full JEPA. Width is 32 for this bounded timing probe. This is a systems fixture, not the training profile or a parameter-matched representation study.

Two warm-ups precede five timed repeats. Forward runs without autograd; forward/backward includes graph construction and backward but excludes optimizer/EMA. CUDA, when selected, synchronizes correctly. Rows save all samples, median, MAD, backend events and actual parameter counts. Geometric cache construction is amortized by warm-up; per-step feature moments are rebuilt. Geometry caches are cleared between benchmark cases.

CPU peak tensor memory is explicitly NOT_RUN. CUDA records baseline and total/incremental peak live allocated bytes; parameters are included in total peak, cached reservation is excluded. This prototype's Python launches, indexing, temporary edges and diagnostic scalar synchronizations are known costs. Other machine workloads are not controlled. Slower FMM measurements are valid findings; finite sweeps do not prove asymptotic exponents.

In [ ]:
benchmark_config = config["benchmark"]
# Always retain a small CPU baseline; the optional panel below is separately labeled.
benchmark_rows = benchmark_sweep([(4,4),(8,8),(12,16)], device="cpu", scopes=tuple(benchmark_config["scopes"]),
                                 dim=32, warmup=2, repeats=5, batch_size=1)
write_csv(ARTIFACT/"scaling_metrics.csv", benchmark_rows)
atomic_json(ARTIFACT/"scaling_metrics.json", benchmark_rows)
save_figure(benchmark_figure(benchmark_rows), ARTIFACT, "cpu_scaling")
print("CPU rows:", len(benchmark_rows), "Largest CPU grid:", (12,16))

### Optional single-GPU panel

This optional panel runs one small rectangular float32 JEPA optimizer/EMA step and compares mixers at 196 and 784 tokens with width 32 and batch 1. It neither consumes all available GPUs nor launches the larger GPU training profile. CUDA exact-resume and mixed precision remain untested here. Hardware-specific measurements are optional; required CPU correctness gates still determine readiness.

In [ ]:
if os.environ.get("FMM_JEPA_OPTIONAL_CUDA", "0") == "1":
    cuda_results = cuda_smoke_test(config)
    if cuda_results["optional_cuda"]["status"] == "PASS":
        gpu_rows = benchmark_sweep([(14,14),(28,28)], device="cuda:0", scopes=("mixer",), dim=32, warmup=2, repeats=5, batch_size=1)
        benchmark_rows.extend(gpu_rows)
        write_csv(ARTIFACT/"scaling_metrics.csv", benchmark_rows)
        atomic_json(ARTIFACT/"scaling_metrics.json", benchmark_rows)
    else: gpu_rows=[]
else:
    cuda_results = {"optional_cuda":{"status":"NOT_RUN","details":"Set FMM_JEPA_OPTIONAL_CUDA=1 to run the bounded optional panel"}}
    gpu_rows=[]
atomic_json(ARTIFACT/"cuda_certificates.json", cuda_results)
print(json.dumps(cuda_results, indent=2))
for row in gpu_rows:
    print(row["kind"], row["N_grid"], row["mode"], round(row["median_ms"],3), "ms; peak allocated", row["peak_allocated_bytes"], "bytes;", row["backend"])

## 13. Short synthetic optimization and resumable checkpoint

Twelve changing-mask steps are the default CPU smoke. Images combine low-frequency structure and noise, generated from the checkpointed stream. AdamW updates context/predictor; only a successful step triggers EMA. Log loss, representation dispersion/norms, near/far norms, gradient norms and RSS/current CUDA allocation observations. A moving teacher and changing masks do not imply monotone loss. Dispersion does not establish the absence of representation collapse, and synthetic loss does not measure representation quality.

In [ ]:
trained_model, training_metrics, resume_status = run_synthetic(config, CHECKPOINT_DIR, SOURCE_HASHES)
write_csv(ARTIFACT/"synthetic_training_metrics.csv", training_metrics)
atomic_json(ARTIFACT/"synthetic_training_metrics.json", training_metrics)
atomic_json(ARTIFACT/"resume_status.json", resume_status)
save_figure(training_figure(training_metrics), ARTIFACT, "synthetic_trace")
print("Actual steps:", len(training_metrics), "first loss:", training_metrics[0]["loss"], "last loss:", training_metrics[-1]["loss"])
print(json.dumps(resume_status, indent=2))
smoke_results={"synthetic_optimization":{"status":"PASS" if len(training_metrics)==config["steps"] and all(math.isfinite(row["loss"]) for row in training_metrics) else "FAIL",
                                         "details":{"steps":len(training_metrics),"purpose":"synthetic_smoke_test","metrics":"synthetic_training_metrics.csv"}}}

## 14. Parameter accounting and artifact compatibility

Student and predictor are counted once each. Teacher parameters are non-trainable; buffers are counted separately. The comparison ViT uses the same profile dimensions but has a different parameter count. Later protocols must approximately match parameters or explicitly report their difference. A checkpoint carries schema/configuration/source hashes; fresh initialization remains the Notebook 01 default.

In [ ]:
counts={"fmm":parameter_counts(trained_model)}
baseline_config=copy.deepcopy(config["model"]);baseline_config["kind"]="vit"
baseline_model=JEPA(baseline_config)
counts["vit"]=parameter_counts(baseline_model)
del baseline_model
atomic_json(ARTIFACT/"parameter_counts.json",counts)
checkpoint_manifest=validate_manifest(CHECKPOINT_DIR,config,SOURCE_HASHES)
(ARTIFACT/"checkpoints").mkdir(exist_ok=True)
for label,filename in (("synthetic_smoke_test.pt",checkpoint_manifest["latest"]),("best_synthetic_smoke_test.pt",checkpoint_manifest["best"])):
    link=ARTIFACT/"checkpoints"/label
    if link.is_symlink():link.unlink()
    if not link.exists():link.symlink_to(os.path.relpath(CHECKPOINT_DIR/filename,link.parent))
print(json.dumps(counts,indent=2))

## 15. Goal → Conditions → Certificates → Closure

The goal is a trustworthy core for a real-data learning pilot. Conditions are the named correctness/recovery requirements; certificates link test entry points to saved evidence. Closure is computed from every required gate and the expected registry—missing gates cannot disappear from the decision. Optional CUDA measurements and unavailable CPU peak memory are reported separately. A FAIL or NOT_RUN required gate blocks readiness.

The handoff is an evidence snapshot, not a permanent PASS file. Notebook 01 must check versions/source hashes and rerun compact core tests before training.

In [ ]:
required_names = [
 "geometry_rectangular_coordinates","hierarchy_pair_ownership","interpolation_transfer_identities","classical_field_approximation",
 "near_far_information_flow","padding_zero_contribution","core_gradchecks","fmm_no_allpairs","deterministic_core",
 "patch_shapes_coordinates_rectangular","mask_validity_bounded_fallback","hidden_pixel_leakage","padding_no_messages",
 "context_predictor_target_alignment_loss","finite_forward_backward_student_update","teacher_initialization_and_ema",
 "deterministic_model_correctness","optimized_attention_backend","checkpoint_exact_resume","synthetic_optimization"]
all_results={**geometry_results,**model_results,**checkpoint_results,**smoke_results,**cuda_results}
ledger=[]
for name in required_names+["optional_cuda"]:
    result=all_results.get(name,{"status":"NOT_RUN","details":"Required gate did not return a result"})
    if name in geometry_results: evidence,test="geometry_certificates.json","fmm_tests.test_geometry.run_geometry_tests"
    elif name in model_results: evidence,test="model_certificates.json","fmm_tests.test_models.run_model_tests"
    elif name in checkpoint_results: evidence,test="checkpoint_certificates.json","fmm_jepa.training.checkpoint_resume_test"
    elif name=="optional_cuda": evidence,test="cuda_certificates.json","fmm_jepa.evaluation.cuda_smoke_test"
    else:evidence,test="synthetic_training_metrics.csv","fmm_jepa.training.run_synthetic"
    ledger.append({"goal":"Reusable verified FMM-JEPA core","condition":name,"certificate":evidence,"test":test,
                   "required":name in required_names,"status":result["status"],"details":result["details"],
                   "closure":"satisfied" if result["status"]=="PASS" else "open"})
ready = all(row["status"]=="PASS" for row in ledger if row["required"]) and len([row for row in ledger if row["required"]])==len(required_names)
atomic_json(ARTIFACT/"certificate_ledger.json",{"schema_version":ARTIFACT_SCHEMA,"gates":ledger,"ready_for_notebook_01":ready})
from collections import Counter
print("Required gate statuses:",dict(Counter(row["status"] for row in ledger if row["required"])))
print("Ready for Notebook 01:",ready)
for row in ledger:print(row["status"],row["condition"],"->",row["certificate"])

## 16. Established results, limitations and Notebook 01 handoff

Established results are restricted to the executed gates, fixed classical fixtures, short synthetic run and bounded measurements. Useful representation quality, better quality–cost trade-offs than ViT, whole-system asymptotic superiority, real-dataset accuracy, generalization and novelty are **not established**.

Notebook 01's bounded next task is to verify a Tiny ImageNet adapter and splits, preregister matched FMM/ViT small pilots with fresh initialization, verify resumability, evaluate frozen representations and export a new handoff. For 64×64 source images, patch size 8 is a starting candidate. Labels are used only for the declared frozen probe. No real-data run starts here.

See [experiment protocol](../docs/experiment_protocol.md), [architecture](../docs/architecture.md) and [source review](../docs/references.md). The next cell saves the machine-readable compatibility contract and the actual engineering report.

In [ ]:
limitations=[
 "Useful representation learning, real-dataset scores, generalization, novelty and a quality-compute advantage are not established.",
 "Prototype uses Python-level hierarchy/edge operations and synchronous diagnostic scalar reads; no custom fused FMM kernels.",
 "Allocated empty square-tree cells count toward cost; very thin rectangular grids can have substantial padding overhead.",
 "FMM and ViT widths match in this smoke, but actual parameter counts differ; future comparisons must register budgets.",
 "CPU peak tensor memory was not measured. CUDA memory is live allocator accounting, not total device/process memory.",
 "Mixed precision, distributed execution and CUDA bitwise resume were not tested; CPU float32 resume was tested.",
 "Bounded classical interpolation results do not transfer to learned semantic translations.",
 "Conventional Transformer predictor remains an attention cost; no whole-JEPA linear-time claim.",
 "Boot disk instance-deletion policy is unverified; checkpoints have not been copied off-instance."]
tested=["cpu/float32 neural and recovery", "cpu/float64 classical identities and gradchecks"]
if cuda_results["optional_cuda"]["status"]=="PASS":tested.append("cuda:0/float32 small optimizer/EMA smoke and mixer profiling")
handoff={"schema_version":ARTIFACT_SCHEMA,"package_version":"0.1.0","repository_commit":ENVIRONMENT["repository_commit"],
 "config_schema_version":CONFIG_SCHEMA,"run_id":RUN_ID,"source_hashes":SOURCE_HASHES,"configuration_hash":config_hash(config),
 "artifact_directory":str(ARTIFACT),"ready_for_notebook_01":ready,"required_gates":[row for row in ledger if row["required"]],
 "import_contract":"docs/notebook_imports.md; execute only bootstrap ahr-export definitions, then install_notebook_package(root)",
 "model_example":"from fmm_jepa.models import JEPA; model = JEPA(config['model'])",
 "interfaces":{"images":"float BCHW, RGB, dimensions divisible by patch_size","patch_features":"B,V,D",
   "indices":"B,V int64 original row-major indices; unique per valid row","valid":"B,V bool; each sample/block has at least one valid token; padding sentinel -1; padding contributes no messages",
   "masking":"target blocks may intentionally overlap; context strictly excludes their union; bounded retries plus documented anchor fallback",
   "coordinates":"((col+0.5)/grid_cols, (row+0.5)/grid_rows)","encoder":"features in requested index order; rectangular positive grids",
   "predictor":"list of B,T_k,D outputs aligned to each requested target block; equal-sample and equal-block weighting; valid tokens averaged within each sample/block",
   "dataset":"image, sample_id, split, source_dimensions, optional labels/annotations; labels excluded from JEPA loss"},
 "checkpoint":{"purpose":"synthetic_smoke_test","directory":str(CHECKPOINT_DIR),"manifest":checkpoint_manifest,
   "compatibility":"config/source plus saved Python/PyTorch/CUDA/device/determinism settings checked on restore; same physical hardware remains a caller requirement; initialize fresh for Notebook 01","resume_cell":"run-synthetic-smoke"},
 "tested_device_dtype":tested,"untested":["mixed precision","distributed training","CUDA exact resume"],"limitations":limitations,
 "baseline":"torch.nn.functional.scaled_dot_product_attention; actual dispatch and parameters in scaling_metrics.csv",
 "benchmark_protocol":{"scopes":["mixer","encoder_block","full_jepa"],"masks":"same seeds/workloads per pair","warmup":2,"repeats":5,
   "timing":"median/MAD plus raw forward and forward_backward samples; no optimizer/EMA in timing","memory":"see each raw row's memory_definition"},
 "tiny_imagenet_adapter_requirements":["pin and verify Hugging Face repository and immutable revision","verify schema, license/access conditions and source dimensions",
   "register deterministic train/development/evaluation split IDs; prevent image duplication leakage","keep final/lockbox evaluation separate from development",
   "check all class IDs and label mapping; labels excluded from SSL"],
 "proposed_pilot":{"source_resolution":[64,64],"patch_size":8,"initialization":"fresh independently for FMM and ViT",
   "task":"frozen-encoder single-label linear probe with registered accuracy metric", "seeds":"register before launch and report actual completed count",
   "budget":"small matched pilot only; choose steps/batch after measured memory and learning checks","checkpoint":"verify exact resume before any >15-minute run"},
 "next_task":"Notebook 01: dataset adapter, small registered FMM/ViT pilots, frozen-representation evaluation, resume verification, next handoff; NOT EXECUTED",
 "entry_checks":["reject schema/config/source incompatibilities","rerun fmm_tests.test_geometry.run_geometry_tests and fmm_tests.test_models.run_model_tests",
                 "verify checkpoint resume on chosen backend before long runs","do not load synthetic smoke weights for default comparison"]}
atomic_json(ARTIFACT/"handoff_00_to_01.json",handoff)
atomic_json(PROJECT_ROOT/"handoff_00_to_01.json",handoff)
manifest={"schema_version":ARTIFACT_SCHEMA,"run_id":RUN_ID,"completed_at_utc":datetime.datetime.now(datetime.timezone.utc).isoformat(),
 "status":"completed","configuration_hash":config_hash(config),"source_hashes":SOURCE_HASHES,"environment":ENVIRONMENT,
 "data_source":"synthetic","generation":"fmm_jepa.data.synthetic_images; checkpointed seeded generator; labels absent",
 "seed":config["seed"],"precision":config["dtype"],"profile":PROFILE,"effective_global_batch":config["batch_size"]*config["world_size"]*config["gradient_accumulation"],
 "checkpoint_manifest":str(CHECKPOINT_DIR/"manifest.json"),"required_status_counts":dict(Counter(row["status"] for row in ledger if row["required"])),
 "ready_for_notebook_01":ready,"foundation_status":"verified" if ready else "partially verified","research_hypothesis_status":"not yet established"}
atomic_json(ARTIFACT/"run_manifest.json",manifest)
rows_report=["# Notebook 00 engineering report", "", "Current milestone: reusable core and verification.",
 "", "Notebook: `notebooks/00_FMM_JEPA_Core_Theory_and_Verification.ipynb`.",
 "", "Clean execution: this run reached its final cell; the external `clean_execution.json` records completed clean-kernel validation.",
 "", "Device and precision: "+", ".join(tested)+".",
 "", "Modules: bootstrap, geometry, operators, models, data, training, evaluation and reporting, all tagged notebook modules.",
 "", "| Model | Context | Predictor | Total trainable | Frozen teacher |", "|---|---:|---:|---:|---:|"]
for kind,count in counts.items():rows_report.append(f"| {kind.upper()} | {count['context']:,} | {count['predictor']:,} | {count['trainable']:,} | {count['target']:,} |")
rows_report += ["", f"Required gates: **{manifest['required_status_counts']}**. Gradient isolation and EMA: {model_results['teacher_initialization_and_ema']['status']}; exact CPU checkpoint/resume: {checkpoint_results['checkpoint_exact_resume']['status']}.",
 "", "| Classical grid | Order p | Absolute L2 error | Stabilized relative L2 error |", "|---|---:|---:|---:|"]
for case in classical_cases:rows_report.append(f"| {case['grid'][0]} × {case['grid'][1]} | {case['order']} | {case['absolute_l2_error']:.6g} | {case['stabilized_relative_l2_error']:.6g} |")
rows_report += ["", f"Synthetic optimization: {len(training_metrics)} steps; first loss {training_metrics[0]['loss']:.6f}, last loss {training_metrics[-1]['loss']:.6f}. This is a software smoke test.",
 "", f"Largest measured grid: **{max(benchmark_rows,key=lambda r:r['N_grid'])['rows']} × {max(benchmark_rows,key=lambda r:r['N_grid'])['cols']}**. Timing below is forward + backward; MAD is repeat dispersion. Full raw rows include all sizes and forward-only results.",
 "", "| Device / scope | Grid tokens | Model | Median ms | MAD ms | Peak allocated MiB |", "|---|---:|---|---:|---:|---:|"]
for row in benchmark_rows:
    if row["mode"]=="forward_backward" and (row["N_grid"]==192 or row["N_grid"]==784):
        memory=f"{row['peak_allocated_bytes']/2**20:.2f}" if row['peak_allocated_bytes'] is not None else "not measured"
        rows_report.append(f"| {row['device']} / {row['scope']} | {row['N_grid']} | {row['kind'].upper()} | {row['median_ms']:.3f} | {row['mad_ms']:.3f} | {memory} |")
compared=[(row,next(other for other in benchmark_rows if other['kind']=='vit' and all(other[key]==row[key] for key in ('device','scope','mode','N_grid')))) for row in benchmark_rows if row['kind']=='fmm']
fmm_slower_count=sum(a['median_ms']>b['median_ms'] for a,b in compared)
observed_backends=sorted({row['device']+": "+row['backend'] for row in benchmark_rows if row['kind']=='vit'})
rows_report += ["", f"FMM is slower in {fmm_slower_count}/{len(compared)} measured comparisons. Observed baseline backends: {'; '.join(observed_backends)}. This is an implementation measurement, not a quality–compute comparison or asymptotic proof.",
 "", "Known bottlenecks and limitations:"]+["- "+item for item in limitations]+["", f"Artifact directory: `{ARTIFACT}`.",
 "", f"Ready for Notebook 01: **{'YES — all required gates passed' if ready else 'NO — inspect required failed/unexecuted gates'}**.",
 "", "Next task: Notebook 01 — Tiny ImageNet learning pilot.",
 "", f"**Foundation status: {manifest['foundation_status']}.**", "", "**Research hypothesis status: not yet established**; learning and comparative evidence belong to later notebooks."]
report="\n".join(rows_report)
(ARTIFACT/"engineering_report.md").write_text(report+"\n")
(PROJECT_ROOT/"docs/notebook_01_handoff.md").write_text("# Notebook 01 handoff\n\n"+report+"\n\nMachine contract: [handoff_00_to_01.json](../handoff_00_to_01.json).\n\n"+"```json\n"+json.dumps({"entry_checks":handoff["entry_checks"],"adapter_requirements":handoff["tiny_imagenet_adapter_requirements"],"pilot":handoff["proposed_pilot"]},indent=2)+"\n```\n")
display(Markdown(report))